# Example-30: Frequency estimation from complex data

In [1]:
# In this example frequency estimation is compared for real & complex data
# Complex normalized data is obtained using transport matrices between monitor locations
# Momentum at given location is computed using model transport matrix between given location and the next one
# Model normalization matrix is used to compute normalized data

In [2]:
# Import

import numpy
import torch
import nufft
import yaml

import sys
sys.path.append('..')

from harmonica.util import mod
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.decomposition import Decomposition
from harmonica.filter import Filter
from harmonica.model import Model
from harmonica.parameterization import cs_normal

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())
print(torch.get_num_threads())

True
16


In [3]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [4]:
# Set error & noise amplitude

error = 1.0E-3
noise = 1.0E-5

In [5]:
# Generate test trajectories

# Set model

model = Model(path='../config.yaml', model='uncoupled', dtype=dtype, device=device)

# Generate and apply errors

model.make_error(1.0*error, 0.0E-3)

# Compute transport matrices

model.make_transport(error=True, exact=False)

# Compute twiss

flag = model.make_twiss()
print(flag)

# Set initial condition

initial = torch.tensor([0.002, 0.0, 0.002, 0.0], dtype=dtype, device=device)

# Set number of iterations

length = 2**12

# Generate trajectories at monitor locations

trajectory =  model.make_trajectory(initial, length, error=True, transport=True)[model.monitor_index]
print(trajectory.shape)

True
torch.Size([54, 4096, 4])


In [6]:
# Model tunes

print(mod(torch.stack([model.nux, model.nuy]), 1.0))

# Fractional tunes with effect of errors

mod(model.out_tune, 1.0)

tensor([5.368830987374e-01, 5.767746333258e-01], dtype=torch.float64)


tensor([5.355176078608e-01, 5.859099108267e-01], dtype=torch.float64)

In [7]:
# Set data length

length = 2**10

# Set coordinates

qx = trajectory[..., 0]
qy = trajectory[..., 2]

# Add noise

qx += 1.0*noise*torch.rand_like(qx)
qy += 1.0*noise*torch.rand_like(qy)

# Get momenta

px = []
py = []

index_i = torch.tensor(model.monitor_index, dtype=torch.int64, device=device)
index_j = index_i.roll(-1)
for index, (i, j) in enumerate(zip(index_i, index_j)):
    i, j = int(i), int(j)
    matrix = model.matrix(i, j)  if i < j else model.matrix(i, j + model.size)
    qxi = qx[index, :length]
    qxj = qx[int(mod(index + 1, 54)), :length] if i < j else qx[int(mod(index + 1, 54)), 1:length + 1]
    qyi = qy[index, :length]
    qyj = qy[int(mod(index + 1, 54)), :length] if i < j else qy[int(mod(index + 1, 54)), 1:length + 1]
    pxi, pyi = model.make_momenta(matrix, qxi, qxj, qyi, qyj)
    px.append(pxi)
    py.append(pyi)
    
qx = qx[:, :length]
px = torch.stack(px)
qy = qy[:, :length]
py = torch.stack(py)

# Set normalized coordinates

qnx, pnx, qny, pny = [], [], [], []

for i, (qxi, pxi, qyi, pyi) in enumerate(zip(qx, px, qy, py)):
    index = model.monitor_index[i]
    normal = cs_normal(model.ax[index], model.bx[index], model.ay[index], model.by[index])
    qnxi, pnxi, qnyi, pnyi = normal.inverse() @ torch.stack([qxi, pxi, qyi, pyi])
    qnx.append(qnxi)
    pnx.append(pnxi)
    qny.append(qnyi)
    pny.append(pnyi)
    
qnx = torch.stack(qnx)
pnx = torch.stack(pnx)
qny = torch.stack(qny)
pny = torch.stack(pny)

# Set complex coordinates

cx = qnx - 1j*pnx
cy = qny - 1j*pny

In [8]:
# Estimate frequency from real data

win = Window.from_cosine(length, 1.0)

tbt = Data.from_data(win, qx)
fre = Frequency(tbt, f_range=(0.0, 0.5))
tbt.window_remove_mean()
tbt.window_apply()
fre('parabola')
fx_real = torch.clone(1.0 - fre.frequency)
tbt.reset()
fx_candan_real = 1.0 - fre.candan_get_frequency()

tbt = Data.from_data(win, qy)
fre = Frequency(tbt, f_range=(0.0, 0.5))
tbt.window_remove_mean()
tbt.window_apply()
fre('parabola')
fy_real = torch.clone(1.0 - fre.frequency)
tbt.reset()
fy_candan_real = 1.0 - fre.candan_get_frequency()

In [9]:
# Estimate frequency from complex data

win = Window.from_cosine(length, 1.0)

tbt = Data.from_data(win, cx)
fre = Frequency(tbt, f_range=(0.0, 1.0))
tbt.window_remove_mean()
tbt.window_apply()
fre('parabola')
fx_complex = torch.clone(fre.frequency)
tbt.reset()
fx_candan_complex = fre.candan_get_frequency()

tbt = Data.from_data(win, cy)
fre = Frequency(tbt, f_range=(0.0, 1.0))
tbt.window_remove_mean()
tbt.window_apply()
fre('parabola')
fy_complex = torch.clone(fre.frequency)
tbt.reset()
fy_candan_complex = fre.candan_get_frequency()

In [10]:
# Compare results

# Real

print(torch.stack([fx_real.mean(), fy_real.mean()]) - mod(model.out_tune, 1.0))
print(torch.stack([fx_real.std(), fy_real.std()]))
print()

# Complex

print(torch.stack([fx_complex.mean(), fy_complex.mean()]) - mod(model.out_tune, 1.0))
print(torch.stack([fx_complex.std(), fy_complex.std()]))
print()

tensor([4.023446575907e-09, 8.119718120092e-09], dtype=torch.float64)
tensor([4.510849132034e-08, 6.335469004285e-08], dtype=torch.float64)

tensor([1.534227411781e-09, 1.144999683955e-08], dtype=torch.float64)
tensor([6.337156885518e-08, 6.110344538303e-08], dtype=torch.float64)



In [11]:
# Without errors & noise, frequency estimation is better for complex data (more accurate and less spread)
# With errors, accuracy is close, but spread is smaller for complex data
# With error & noise results are similar (for signals with only one component)

In [12]:
# Candan (real)

print(torch.stack([fx_candan_real.mean(), fy_candan_real.mean()]) - mod(model.out_tune, 1.0))
print(torch.stack([fx_candan_real.std(), fy_candan_real.std()]))
print()

# Candan (complex)

print(torch.stack([fx_candan_complex.mean(), fy_candan_complex.mean()]) - mod(model.out_tune, 1.0))
print(torch.stack([fx_candan_complex.std(), fy_candan_complex.std()]))
print()

tensor([-1.167521745948e-09,  9.240365583807e-09], dtype=torch.float64)
tensor([7.403972533258e-08, 5.187452987589e-08], dtype=torch.float64)

tensor([2.182026559439e-09, 1.332983401436e-08], dtype=torch.float64)
tensor([7.544953254239e-08, 4.869088182028e-08], dtype=torch.float64)

